In [1]:
import os
import glob
from datasets import load_dataset

# 诊断代码
general_path = "/DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--Fineweb-Edu-Chinese-V2_1-subset-5M/snapshots/a93f033ad83602bc35a03a49e6199dde5bcd1455/data"

print("=== 诊断信息 ===")

# 1. 检查目录是否存在
print(f"目录是否存在: {os.path.exists(general_path)}")

# 2. 列出所有parquet文件
parquet_files = glob.glob(f"{general_path}/train-*.parquet")
print(f"找到的parquet文件数量: {len(parquet_files)}")

# 3. 检查前几个文件的状态
for i, file_path in enumerate(parquet_files[:3]):  # 只检查前3个文件
    print(f"\n文件 {i+1}: {os.path.basename(file_path)}")
    print(f"  - 文件存在: {os.path.exists(file_path)}")
    print(f"  - 文件大小: {os.path.getsize(file_path) if os.path.exists(file_path) else 'N/A'} bytes")
    print(f"  - 是否为软链接: {os.path.islink(file_path)}")
    
    if os.path.islink(file_path):
        link_target = os.readlink(file_path)
        print(f"  - 链接目标: {link_target}")
        
        # 检查绝对路径的链接目标
        if not os.path.isabs(link_target):
            abs_target = os.path.join(os.path.dirname(file_path), link_target)
            print(f"  - 绝对链接目标: {abs_target}")
            print(f"  - 链接目标存在: {os.path.exists(abs_target)}")

# 4. 尝试用pandas直接读取一个文件
try:
    import pandas as pd
    if parquet_files:
        test_file = parquet_files[0]
        print(f"\n=== 尝试用pandas读取第一个文件 ===")
        df = pd.read_parquet(test_file)
        print(f"成功读取，数据形状: {df.shape}")
        print(f"列名: {list(df.columns)}")
        if len(df) > 0:
            print(f"第一行数据: {df.iloc[0].to_dict()}")
        else:
            print("文件为空")
except Exception as e:
    print(f"pandas读取失败: {e}")

/home/chenyuhang/anaconda3/envs/transformers/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== 诊断信息 ===
目录是否存在: True
找到的parquet文件数量: 7

文件 1: train-00005-of-00007.parquet
  - 文件存在: True
  - 文件大小: 2240710120 bytes
  - 是否为软链接: True
  - 链接目标: ../../../blobs/16fc4c947a53e396bd6abecc034cf4303d04d7398a59c138940e7bf6be775d20
  - 绝对链接目标: /DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--Fineweb-Edu-Chinese-V2_1-subset-5M/snapshots/a93f033ad83602bc35a03a49e6199dde5bcd1455/data/../../../blobs/16fc4c947a53e396bd6abecc034cf4303d04d7398a59c138940e7bf6be775d20
  - 链接目标存在: True

文件 2: train-00002-of-00007.parquet
  - 文件存在: True
  - 文件大小: 2197349790 bytes
  - 是否为软链接: True
  - 链接目标: ../../../blobs/fb620fd127baeb271656108ee5b6b205b1c62a6cd4c8e2d40149f5f971785acc
  - 绝对链接目标: /DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--Fineweb-Edu-Chinese-V2_1-subset-5M/snapshots/a93f033ad83602bc35a03a49e6199dde5bcd1455/data/../../../blobs/fb620fd127baeb271656108ee5b6b205b1c62a6cd4c8e2d40149f5f971785acc
  - 链接目标存在: True

文件 3: train-00004-of-00007.parquet
  - 文件存在: True
  - 文件大小: 2180313

In [2]:
import os
from datasets import load_dataset

def resolve_symlinks(file_pattern):
    """解析软链接并返回实际文件路径"""
    import glob
    files = glob.glob(file_pattern)
    resolved_files = []
    
    for file_path in files:
        if os.path.islink(file_path):
            # 解析软链接
            link_target = os.readlink(file_path)
            if not os.path.isabs(link_target):
                # 相对路径，转换为绝对路径
                abs_target = os.path.join(os.path.dirname(file_path), link_target)
            else:
                abs_target = link_target
            
            if os.path.exists(abs_target):
                resolved_files.append(abs_target)
            else:
                print(f"警告: 软链接目标不存在: {abs_target}")
        else:
            resolved_files.append(file_path)
    
    return resolved_files

# 使用解析后的文件路径
general_path = "/DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--Fineweb-Edu-Chinese-V2_1-subset-5M/snapshots/a93f033ad83602bc35a03a49e6199dde5bcd1455/data"
resolved_files = resolve_symlinks(f"{general_path}/train-*.parquet")

print(f"解析后的文件数量: {len(resolved_files)}")

if resolved_files:
    ds = load_dataset(
        "parquet",
        data_files=resolved_files,
        split="train"
    )
    print(f"成功加载数据集，大小: {len(ds)}")
else:
    print("没有找到有效的parquet文件")

解析后的文件数量: 7


Generating train split: 4956057 examples [00:54, 90831.25 examples/s] 

成功加载数据集，大小: 4956057


In [3]:
ds

Dataset({
    features: ['text', 'source'],
    num_rows: 4956057
})

In [7]:
# 统计第一条数据的字段长度
first_data = ds[0]
text_length = len(first_data['text'])
text_length


3629

In [8]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B",
                                          use_fast = True)

In [9]:
def tokenize_function(examples):
    return tokenizer(examples['text'],
                     max_length=2048,
                     truncation=True,
                     padding="max_length")

tokenizer_dataset = ds.map(tokenize_function,
                                num_proc=40,
                                batched=True, 
                                remove_columns=['text']) #! 移除原始文本节省内存

Map (num_proc=40): 100%|██████████| 4956057/4956057 [05:09<00:00, 16032.56 examples/s]


In [10]:
tokenizer_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

In [11]:
tokenizer_dataset

Dataset({
    features: ['source', 'input_ids', 'attention_mask'],
    num_rows: 4956057
})

In [12]:
tokenizer_dataset.save_to_disk("/DATA/disk2/yuhang/.cache/bit_brain_data/Annealing/Fineweb-Edu-Chinese-V2_1-subset-5M",
                               max_shard_size = "2024MB")

Saving the dataset (26/26 shards): 100%|██████████| 4956057/4956057 [00:35<00:00, 141160.12 examples/s]
